# Notebook 02 – EDA after Cohort Extraction


--------

Cohort Population Extracted from 01 notebook

**Main Objectives**

+ Verify that data missingness reflects real-world medical patterns
+ Verify assumptions based off feature distributions and sample sizes
+ Baseline metrics are established and able to be used


In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import missingno as msno
import duckdb

from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.metrics import roc_auc_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi' : 300})

sys.path.insert(0, '../src')



In [2]:
from icu_tft.data.connect import get_connection

SQL_PATH = '../src/icu_tft/data/sql/mimic_iv_24h_icu_mortality_cohort.sql'
COHORT_PQ = '../data/processed/cohort.parquet'

con = get_connection()
with open(SQL_PATH) as f:
    cohort_sql = f.read()
    
con.execute(f'CREATE OR REPLACE TABLE cohort AS (\n{cohort_sql}\n)')
print('cohort table written to duckdb')

cohort = con.execute('SELECT * FROM cohort').df()
cohort.to_parquet(COHORT_PQ, index=False)

print(f'Cohort saved to {COHORT_PQ} shape={cohort.shape}')
print(cohort[['stay_id', 'mortality_24h', 'mortality_inhospital']].head())
    

[connect.py] Registered 30 views against mimic.duckdb
[connect.py] WARNING — 1 file(s) not found (views skipped):
  /Users/longer/ICU_Mortality_Prediction/data/raw/hosp/antimicrobial.csv.gz
cohort table written to duckdb
Cohort saved to ../data/processed/cohort.parquet shape=(67223, 14)
    stay_id  mortality_24h  mortality_inhospital
0  37081114              0                     0
1  37067082              0                     0
2  31205490              0                     0
3  37510196              0                     1
4  39060235              0                     0


In [3]:

# architecture for synthetic data generation matching preset schema
# np.random.seed(617)
# n_patients = 1000
# stay_ids = np.arange(10000, 10000 + n_patients)
# cohort = pd.DataFrame(
#     {
#         'stay_id' : stay_ids,
#         'age' : np.random.normal(65, 15, n_patients).clip(18,100),
#         'gender' : np.random.choice(['M', 'F'], n_patients),
#         'race' : np.random.choice(['White', 'Black', 'Hispanic', 'Asian', 'Other', 'Unknown'], n_patients),
#         'insurance' : np.random.choice(['Medicare', 'Medicaid', 'Private', 'Other'], n_patients),
#         'mortality_24h' : np.random.binomial(1, 0.15, n_patients)
#     }
# )

# ts_records = []

# for sid, outcome in zip(cohort['stay_id'], cohort['mortality_24h']):
#     for t in range(24):
#         hr = np.random.normal(80 + (t*0.5 if outcome else 0), 15)
#         map_ = np.random.normal(85 - (t*0.5 if outcome else 0), 10)
#         lactate = np.random.exponential(1.5 + (t*0.1 if outcome else 0))
#         gcs = np.random.normal(14 - (t*0.2 if outcome else 0), 2).clip(3, 15)        
#         if np.random.rand() < 0.2: hr - np.nan
#         if np.random.rand() < 0.6: lactate - np.nan
        
#         ts_records.append([sid, t, hr, map_, lactate, gcs])
# ts = pd.DataFrame(ts_records, columns=['stay_id', 'time_step', 'heart_rate', 'mbp', 'lactate', 'gcs_total'])


## Cohort Data Flow Representation

In [4]:
base_cohort = con.execute('SELECT stay_id, mortality_24h, mortality_inhospital FROM cohort').df()

static_features_df = pd.read_parquet('../data/processed/static_features.parquet')
ts = pd.read_parquet('../data/processed/timeseries_features.parquet')

cols_to_drop = [c for c in static_features_df.columns if c.endswith('_y')]
static_features_df = static_features_df.drop(columns=cols_to_drop)
static_features_df.columns = static_features_df.columns.str.replace('_x', '', regex=False)
final_static_df = pd.merge(
    static_features_df,
    base_cohort,
    on='stay_id',
    how='inner'
)

print(final_static_df[['stay_id', 'mortality_24h']].head())

    stay_id  mortality_24h
0  37081114              0
1  37067082              0
2  31205490              0
3  37510196              0
4  39060235              0


In [5]:
def draw_cohort_flow():
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.axis('off')
    
    counts = [
        ('Total ICU Stays (MIMIC-IV)', 67223),
        ('Adults (Age >= 18)', 60769),
        ('First ICU Stay Only', 55000),
        ('ICU LOS >= 24h', 67223),
    ]

In [49]:
def create_demographic_table(df, target='mortality_24h'):
    survived = df[df[target] == 0]
    died = df[df[target] == 1]
    
    rows = []
    
    stat, p_val = mannwhitneyu(survived['anchor_age'], died['anchor_age'], alternative='two-sided')
    rows.append({
        'Feature': 'Age (Median [IQR])',
        f'Survived (n={len(survived)})': (
            f"{survived['anchor_age'].median():.1f} "
            f"[{survived['anchor_age'].quantile(0.25):.1f}-"
            f"{survived['anchor_age'].quantile(0.75):.1f}]"
        ),
        f'Died (n={len(died)})': (
            f"{died['anchor_age'].median():.1f} "
            f"[{died['anchor_age'].quantile(0.25):.1f}-"
            f"{died['anchor_age'].quantile(0.75):.1f}]"
        ),
        'P-Value': f'{p_val:.3f}'
    })
    
    cat_vars = ['gender', 'race', 'insurance']
    for var in cat_vars:
        rows.append({
            'Feature': f'**{var.capitalize()}**',
            f'Survived (n={len(survived)})': '',
            f'Died (n={len(died)})': '',
            'P-Value': ''
        })
        contingency = pd.crosstab(df[var], df[target])

        try:
            _, p_val, _, _ = chi2_contingency(contingency)
            p_str = f'{p_val:.3f}'
        except Exception:
            p_str = 'N/A'

        for i, val in enumerate(sorted(df[var].dropna().unique())):
            # FIX 5: was (survived[var] == var) — var is the column name, not the value
            surv_pct = (survived[var] == val).mean() * 100
            died_pct = (died[var]     == val).mean() * 100
            rows.append({
                'Feature': f'  {val}',
                f'Survived (n={len(survived)})': f"{(survived[var] == val).sum()} ({surv_pct:.1f}%)",
                f'Died (n={len(died)})':         f"{(died[var]     == val).sum()} ({died_pct:.1f}%)",
                'P-Value': p_str if i == 0 else ''
            })

    # FIX 6: return was inside the for-loop — dedented to function level
    table_df = pd.DataFrame(rows)
    return table_df


demo_table = create_demographic_table(cohort)

import pathlib
pathlib.Path('reports/figures').mkdir(parents=True, exist_ok=True)
demo_table.to_csv('reports/figures/02_demographics_table.csv', index=False)


## Boundary Checks

**Objs**
• Rule out human errors in data imputation
• Calc. min, max, 1st, 99th percentiles for all continuous variables to flag impossible values for all features


In [52]:
continuous_vars = []
for names in ts.columns:
    if not names.endswith('_missing'):
        continuous_vars.append(names)
        
cols_to_drop = ['stay_id', 'time_step', 'time_bin']

for cols in cols_to_drop:
    continuous_vars.remove(cols)
continuous_vars


['index',
 'heart_rate',
 'sbp',
 'dbp',
 'mbp',
 'spo2',
 'resp_rate',
 'temperature_c',
 'gcs_total',
 'creatinine',
 'bun',
 'sodium',
 'potassium',
 'bicarbonate',
 'lactate',
 'wbc',
 'hemoglobin',
 'platelets',
 'inr']

In [53]:

pd.set_option('display.float_format', lambda x: '%.2f' % x)

boundary_stats = ts[continuous_vars].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
display(boundary_stats[['min', '1%', '50%', '99%', 'max']])

print('=' * 50)
flags = {
    # Vitals
    'heart_rate'   : (ts['heart_rate'] < 20) | (ts['heart_rate'] > 300),
    'sbp'          : (ts['sbp'] < 40) | (ts['sbp'] > 300),
    'dbp'          : (ts['dbp'] < 20) | (ts['dbp'] > 200),
    'mbp'          : (ts['mbp'] < 20) | (ts['mbp'] > 250),
    'spo2'         : (ts['spo2'] < 0) | (ts['spo2'] > 100),  
    'resp_rate'    : (ts['resp_rate'] < 0) | (ts['resp_rate'] > 120),
    'temperature_c': (ts['temperature_c'] < 20.0) | (ts['temperature_c'] > 45.0),
    'gcs_total'    : (ts['gcs_total'] < 3) | (ts['gcs_total'] > 15),
    
    'creatinine'   : (ts['creatinine'] < 0) | (ts['creatinine'] > 30.0),
    'bun'          : (ts['bun'] < 0) | (ts['bun'] > 250),
    'sodium'       : (ts['sodium'] < 100) | (ts['sodium'] > 180),
    'potassium'    : (ts['potassium'] < 1.0) | (ts['potassium'] > 10.0),
    'bicarbonate'  : (ts['bicarbonate'] < 0) | (ts['bicarbonate'] > 60),
    'lactate'      : (ts['lactate'] < 0) | (ts['lactate'] > 30),
    'wbc'          : (ts['wbc'] < 0) | (ts['wbc'] > 200),
    'hemoglobin'   : (ts['hemoglobin'] < 0) | (ts['hemoglobin'] > 30),
    'platelets'    : (ts['platelets'] < 0) | (ts['platelets'] > 2000),
    'inr'          : (ts['inr'] < 0) | (ts['inr'] > 20)
}

suspicious_masks = {}
total_flags = 0

for var, condition in flags.items():
    if var in ts.columns:
        invalid_count = condition.sum()
        suspicious_masks[var] = condition
        
        if invalid_count > 0:
            print(f"{var:<15}: {invalid_count:>5} suspicious records")
            total_flags += invalid_count

print('-' * 50)
print(f"Total Suspicious Records Flagged: {total_flags}")

,min,1%,50%,99%,max
index,0.00,16133.51,806675.50,1597217.49,1613351.00
heart_rate,0.00,48.00,83.00,134.00,295.00
sbp,0.00,76.00,116.00,178.00,265.00
dbp,NaN,NaN,NaN,NaN,NaN
mbp,0.00,49.00,76.00,120.00,250.00
spo2,0.00,88.00,97.00,100.00,100.00
resp_rate,0.00,8.00,18.00,35.00,80.00
temperature_c,26.67,35.50,36.83,38.67,41.44
gcs_total,3.00,3.00,15.00,15.00,15.00
creatinine,0.00,0.30,1.90,11.50,29.70


heart_rate     :   167 suspicious records
sbp            :   152 suspicious records
mbp            :  2258 suspicious records
wbc            :   325 suspicious records
platelets      :    19 suspicious records
--------------------------------------------------
Total Suspicious Records Flagged: 2921


## Temporal Missingness Analysis

**Objs**
• Calc. missingness of values across our specific temporal hourly bins 
    • This is done to verify our assumptions of future feature engineering efforts that coincide with our timeseries forecasting efforts

In [54]:
ts = ts.reset_index()

In [55]:
ts['time_bin'] = pd.cut(ts['time_step'], bins=[0, 6, 12, 18, 23], labels=['0-6h', '6-12h', '12-18h', '18-24h'], include_lowest=True)

missingness = ts.groupby('time_bin', observed=False)[continuous_vars].apply(lambda x: x.isnull().mean() * 100)

print('=' * 50)
print('Missingness by hourly window (%)')
display(missingness.round(2))

missingness.to_csv('reports/figures/02_temporal_missingness.csv')

Missingness by hourly window (%)


,index,heart_rate,sbp,dbp,mbp,spo2,resp_rate,temperature_c,gcs_total,creatinine,bun,sodium,potassium,bicarbonate,lactate,wbc,hemoglobin,platelets,inr
time_bin,,,,,,,,,,,,,,,,,,,
0-6h,0.00,9.87,26.50,100.00,74.61,10.28,10.43,23.13,17.90,80.14,74.10,100.00,100.00,77.56,79.73,66.99,56.02,79.36,65.35
6-12h,0.00,0.30,25.18,100.00,64.43,0.44,0.78,12.35,5.96,77.37,70.91,100.00,100.00,75.67,81.79,66.51,52.10,77.88,67.68
12-18h,0.00,0.21,26.70,100.00,63.63,0.38,0.70,12.71,6.67,79.64,74.25,100.00,100.00,79.94,88.38,72.69,57.24,80.33,76.51
18-24h,0.00,0.27,23.91,100.00,64.07,0.52,0.81,12.10,7.91,82.04,77.65,100.00,100.00,83.19,91.46,77.27,63.70,82.06,81.41


## Collinearity & Observation Volume

**Objs**
+ Looking for high correlation between certain features
+ Flag any paris of feature pairs with a simple Spearman corr. > 0.85
+ Also test if there is a significant difference between survivors and non-survivors



In [56]:
# spearman testing

print('=' * 50)
print('Spearman Testing of Highly Correlated Feature Pairs ( x > 0.85 )')

patient_means = ts.groupby('stay_id')[continuous_vars].mean()
corr_matrix = patient_means.corr(method='spearman')

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
collinear_pairs = upper_tri.stack().reset_index()

collinear_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
high_corr = collinear_pairs[collinear_pairs['Correlation'].abs() > 0.85]

if high_corr.empty:
    print('No highly collinear pairs found.')
else:
    display(high_corr)
    
display(collinear_pairs)
    
collinear_pairs.to_csv('reports/figures/02_temporal_feature_correlation')

Spearman Testing of Highly Correlated Feature Pairs ( x > 0.85 )
No highly collinear pairs found.


,Feature_1,Feature_2,Correlation
0,index,heart_rate,0.00
1,index,sbp,0.00
2,index,mbp,-0.01
3,index,spo2,-0.00
4,index,resp_rate,-0.00
...,...,...,...
115,wbc,platelets,0.29
116,wbc,inr,0.04
117,hemoglobin,platelets,0.15
118,hemoglobin,inr,-0.20


In [ ]:
obs_counts = ts.groupby('stay_id').size().reset_index(name='total_observations')

obs_eval = pd.merge(
    obs_counts,
    final_static_df[['stay_id', 'mortality_24h']],
    on='stay_id',
    how='inner'
)

survived_obs = obs_eval[obs_eval['mortality_24h'] == 0]['total_observations']
died_obs = obs_eval[obs_eval['mortality_24h'] == 1]['total_observations']

stat, p_val = mannwhitneyu(survived_obs, died_obs, alternative='two-sided')
print(f'Mann Whitney p_val for difference in observation volumne: {p_val:.4f}')

Mann Whitney p_val for difference in observation volumne: 1.0000


: 